# EmpowerLens — D17 cascade inputs (Kaggle GPU)

Trains MentalRoBERTa (multilabel, 10 distortions, distorted rows only) on three input versions —
**full text**, **gold sentences only**, **full text with gold sentences marked** — for seeds 42, 1337, 2024
(9 runs), and evaluates each span model twice: with the **gold** sentences and with the **D03 selector's**
sentences. Output: `results_cascade_inputs/` (15 eval files + small checkpoint metadata), ~1.5–3 h on a T4/P100.

**Before running**
1. Settings → **Accelerator: GPU (T4 or P100)**, **Internet: On**.
2. `mental/mental-roberta-base` is gated: accept its terms on huggingface.co, and add a Kaggle secret
   **`HF_TOKEN`** (Add-ons → Secrets) with a read token.
3. The branch below must contain `data/splits_cascade_inputs/` (pushed from the repo).

**Resumable.** Results go straight to `/kaggle/working/results_cascade_inputs/`; re-running the loop cell
skips every (version, seed) whose eval files already exist. Checkpoints live in `/kaggle/temp` and are deleted
after evaluation. When it finishes, download `results_cascade_inputs.zip`, unzip it into
`results/results_cascade_inputs/` in the repo, and run `python -m src.aggregate_cascade_inputs`.

In [ ]:
# 1. HF token, single GPU, clone the repo into /kaggle/temp (not saved with the output), install.
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"[warn] no HF_TOKEN secret ({e}) - mental-roberta-base will fail to download.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # avoid multi-GPU load hangs on T4 x2

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt
!ls data/splits_cascade_inputs && python -c "import torch; print('cuda:', torch.cuda.is_available())"

In [ ]:
# 2. Shared config + helpers (re-run this cell first after any kernel restart).
import os, json, shutil, subprocess, time
from pathlib import Path
os.chdir("/kaggle/temp/empowerlens")

MODEL  = "mental/mental-roberta-base"
TAG    = MODEL.split("/")[-1]
SEEDS  = (42, 1337, 2024)
DATA   = "data/splits_cascade_inputs"
RES    = Path("/kaggle/working/results_cascade_inputs")
CKPT   = Path("/kaggle/temp/ckpt_d17")
RECIPE = "--max-length 512 --batch-size 16 --epochs 12 --deterministic --early-stopping-patience 3"

def sh(cmd):
    print(f"\n$ {cmd}", flush=True)
    t = time.time()
    subprocess.run(cmd, shell=True, check=True)
    print(f"[done in {(time.time() - t) / 60:.1f} min]", flush=True)

def eval_file(row, seed):
    return RES / row / f"eval_{TAG}_multilabel_{seed}.json"

def train(variant, seed, extra=""):
    sh(f"python -m src.train_transformer --task multilabel --model {MODEL} --seed {seed} "
       f"--splits {DATA}/{variant} --out {CKPT}/{variant} {RECIPE} {extra}")
    return CKPT / variant / f"multilabel_{TAG}_{seed}"

def evaluate(ckpt, splits, row):
    sh(f"python -m src.evaluate --checkpoint {ckpt} --splits {splits} --out {RES / row} --allow-distorted-only")

def keep_meta(ckpt, row, seed):
    dst = RES / "checkpoint_meta" / f"{row}_{seed}"
    dst.mkdir(parents=True, exist_ok=True)
    for f in ("meta.json", "thresholds.json", "epoch_history.csv"):
        if (ckpt / f).exists():
            shutil.copy(ckpt / f, dst / f)
print("config ok:", MODEL, SEEDS, RES)

In [ ]:
# 3. Smoke test (~2 min): tiny run on gold crop, evaluated on the selector crop. Writes nothing to RES.
sh(f"python -m src.train_transformer --task multilabel --model {MODEL} --seed 42 --smoke --epochs 1 "
   f"--max-length 128 --splits {DATA}/crop_gold --out /kaggle/temp/smoke")
sh(f"python -m src.evaluate --checkpoint /kaggle/temp/smoke/multilabel_{TAG}_42_smoke "
   f"--splits {DATA}/crop_sel_42 --out /kaggle/temp/smoke_eval --allow-distorted-only")
ev = json.loads(next(Path("/kaggle/temp/smoke_eval").glob("eval_*.json")).read_text())
assert "macro_specificity_10" in ev["splits"]["test"], "evaluate.py is not the D17 version"
print("SMOKE OK - test macro_f1", ev["splits"]["test"]["metrics"]["macro_f1"])
shutil.rmtree("/kaggle/temp/smoke", ignore_errors=True); shutil.rmtree("/kaggle/temp/smoke_eval", ignore_errors=True)

In [ ]:
# 4. Real runs: 3 versions x 3 seeds = 9 trainings, 15 evaluations. Resumable.
for seed in SEEDS:
    # full text
    if not eval_file("full", seed).exists():
        ck = train("full", seed)
        evaluate(ck, f"{DATA}/full", "full"); keep_meta(ck, "full", seed)
        shutil.rmtree(ck, ignore_errors=True)
    # span versions: train once on gold, evaluate with gold AND with this seed's selector picks
    for kind in ("crop", "marked"):
        gold_row, sel_row = f"{kind}_gold", f"{kind}_sel"
        if eval_file(gold_row, seed).exists() and eval_file(sel_row, seed).exists():
            continue
        ck = train(f"{kind}_gold", seed)
        evaluate(ck, f"{DATA}/{kind}_gold", gold_row)
        evaluate(ck, f"{DATA}/{kind}_sel_{seed}", sel_row)
        keep_meta(ck, gold_row, seed)
        shutil.rmtree(ck, ignore_errors=True)
    print(f"===== seed {seed} done =====", flush=True)

missing = [f"{r}/{s}" for r in ("full", "crop_gold", "crop_sel", "marked_gold", "marked_sel")
           for s in SEEDS if not eval_file(r, s).exists()]
print("MISSING:", missing if missing else "none - all 15 eval files present")

In [ ]:
# 5. Quick look + zip for download (only the results folder is in /kaggle/working).
for row in ("full", "crop_sel", "crop_gold", "marked_sel", "marked_gold"):
    vals = []
    for s in SEEDS:
        p = eval_file(row, s)
        if p.exists():
            e = json.loads(p.read_text())["splits"]
            vals.append((e["val"]["metrics"]["macro_f1"], e["test"]["metrics"]["macro_f1"]))
    if vals:
        v, t = zip(*vals)
        print(f"{row:12s} val {sum(v)/len(v):.3f}  test {sum(t)/len(t):.3f}  (n={len(vals)})")
!cd /kaggle/working && rm -f results_cascade_inputs.zip && zip -qr results_cascade_inputs.zip results_cascade_inputs
!du -sh /kaggle/working/results_cascade_inputs && ls /kaggle/working